# Ablation study — ROC-AUC
Identyczny podział i siatki jak w eksperymencie F1. Wybór hiperparametrów według ROC-AUC; F1 przy domyślnym progu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

In [ ]:
df = pd.read_csv('train_u6lujuX_CVtuZ9i.csv')
X = df.drop(columns=['Loan_ID','Loan_Status'])
y = df.Loan_Status.map({'N':0,'Y':1})
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
train = X_train.assign(Loan_Status=y_train)
print('Train:', X_train.shape, '| Test:', X_test.shape)
display(pd.DataFrame({'train':y_train.value_counts(normalize=True), 'test':y_test.value_counts(normalize=True)}))

In [ ]:
display(pd.crosstab(df.Credit_History.fillna(-1), df.Loan_Status))

In [ ]:
from sklearn.base import clone
variants = {'A': list(X.columns), 'B': list(X.drop(columns='Credit_History').columns), 'C': ['Credit_History']}
num_cols_all = ['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
models = {
    'Regresja logistyczna': (LogisticRegression(max_iter=2000,random_state=RANDOM_STATE),True,
                            {'model__C':[.01,.1,1,10],'model__class_weight':[None,'balanced']}),
    'Drzewo decyzyjne': (DecisionTreeClassifier(random_state=RANDOM_STATE),False,
                         {'model__max_depth':[1,2,3,5,None],'model__min_samples_leaf':[1,5,10],
                          'model__class_weight':[None,'balanced']}),
    'Las losowy': (RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=1),False,
                   {'model__n_estimators':[100],'model__max_depth':[3,5,None],
                    'model__min_samples_leaf':[1,5,10],'model__class_weight':[None,'balanced']})
}


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
rows = []
for variant, cols in variants.items():
    num_cols = [c for c in cols if c in num_cols_all]
    cat_cols = [c for c in cols if c not in num_cols_all]
    for name, (estimator, scale, grid) in models.items():
        num_steps = [('imputer', SimpleImputer(strategy='median'))]
        if scale:
            num_steps.append(('scaler', StandardScaler()))
        prep = ColumnTransformer([
            ('num', Pipeline(num_steps), num_cols),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
        pipe = Pipeline([('preprocessor', prep), ('model', clone(estimator))])
        search = GridSearchCV(pipe, grid, scoring={'roc_auc':'roc_auc','f1':'f1','macro':'f1_macro'}, refit='roc_auc', cv=cv, n_jobs=1, error_score='raise')
        search.fit(X_train[cols], y_train)
        pred = search.predict(X_test[cols])
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        print(variant, name, search.best_score_, roc_auc_score(y_test, search.predict_proba(X_test[cols])[:,1]), flush=True)
        rows.append({'Wariant':variant,'Model':name,'ROC-AUC CV':search.best_score_, 'ROC-AUC CV std':search.cv_results_['std_test_roc_auc'][search.best_index_], 'ROC-AUC test':roc_auc_score(y_test, search.predict_proba(X_test[cols])[:,1]), 'F1 CV':search.cv_results_['mean_test_f1'][search.best_index_], 'Macro CV':search.cv_results_['mean_test_macro'][search.best_index_], 'F1 test':f1_score(y_test,pred), 'Macro test':f1_score(y_test,pred,average='macro'), 'F1 N':f1_score(y_test,pred,pos_label=0), 'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred), 'TN':tn,'FP':fp,'FN':fn,'TP':tp,'Parametry':str(search.best_params_)})
results = pd.DataFrame(rows)
display(results)
results.to_csv('ablation_roc_auc.csv',index=False)


In [ ]:
for metric in ['ROC-AUC CV','ROC-AUC test','Macro test']:
    results.pivot(index='Model',columns='Wariant',values=metric).plot.bar(rot=0,ylim=(0,1),title=metric)
    if 'ROC-AUC' in metric: plt.axhline(.5,ls='--',color='gray')
    plt.tight_layout(); plt.show()